# Reddit Text Classification Pipeline - Google Colab Conversion (Merged CSV Version)

This version assumes you already have `reddit_merged_categories.csv`. It does not require `rspct.tsv` or `subreddit_info.csv`. The notebook is split into independent cells so failures in later steps do not force you to redo earlier steps like splitting, vectorization, tokenization, or training.

In [ ]:
# Ensure a clean slate before mounting
!fusermount -u /content/drive 2>/dev/null
import os
import time
time.sleep(1)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/InformationRet")

print("[OK] Drive ready!")
print("Files:", [f.name for f in BASE_DIR.glob("*.csv")])

## Cell 2 - Project paths and config
Put `reddit_merged_categories.csv` into `/content/drive/MyDrive/InformationRet/`.

In [ ]:
# This cell mounts Google Drive and points the notebook to the project folder.
# We do this first so the dataset and saved models can be loaded from Drive.

from pathlib import Path
import os
import json
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

BASE_DIR = Path('/content/drive/MyDrive/InformationRet')
BASE_DIR.mkdir(parents=True, exist_ok=True)

MERGED_PATH = BASE_DIR / 'reddit_merged_categories.csv'
TRAIN_PATH = BASE_DIR / 'train.csv'
VAL_PATH = BASE_DIR / 'val.csv'
TEST_PATH = BASE_DIR / 'test.csv'

NB_MODEL_PATH = BASE_DIR / 'nb_baseline_model.pkl'
LR_MODEL_PATH = BASE_DIR / 'lr_model.pkl'
DISTIL_MODEL_DIR = BASE_DIR / 'distilbert_model'
DISTIL_RESULTS_DIR = BASE_DIR / 'distilbert_results'
BERT_MODEL_DIR = BASE_DIR / 'bert_model'
BERT_RESULTS_DIR = BASE_DIR / 'bert_results'

CACHE_DIR = BASE_DIR / 'hf_cache'
DISTIL_TOKENIZED_DIR = BASE_DIR / 'distilbert_tokenized'
BERT_TOKENIZED_DIR = BASE_DIR / 'bert_tokenized'
METRICS_DIR = BASE_DIR / 'metrics'
METRICS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42 # For reproducibility purposes
TOP_K_VALUES = [1, 3, 5] # For evaluation metrics like Precision@K and Recall@K

print('BASE_DIR =', BASE_DIR)
print('MERGED_PATH =', MERGED_PATH)
print('Merged CSV exists:', MERGED_PATH.exists())

## Cell 3 - Verify merged dataset is visible

In [ ]:
from tqdm import tqdm
import time
from pathlib import Path
import pandas as pd

print('Folder contents:')
folder_items = sorted(BASE_DIR.iterdir())
for p in tqdm(folder_items, desc="Listing", total=len(folder_items)):
    print('-', p.name)
print()

# File check with progress
print('Verifying dataset.')
files_to_check = [MERGED_PATH]
for file_path in tqdm(files_to_check, desc="Checking"):
    if file_path.exists():
        print(f'[OK] {file_path.name}: OK')
    else:
        print(f'[ERROR] {file_path.name}: MISSING')

assert MERGED_PATH.exists(), f'Missing merged dataset: {MERGED_PATH}'

# Loading with progress bar
print(f'Loading {MERGED_PATH.name}.')
chunksize = 50000
chunks = []
total_rows = 0

# Show loading progress
for chunk in tqdm(pd.read_csv(MERGED_PATH, chunksize=chunksize), desc="Loading", unit="chunk"):
    chunks.append(chunk)
    total_rows += len(chunk)

# Combine + final stats
df = pd.concat(chunks, ignore_index=True)
print(f'[OK] Loaded merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'[OK] Categories: {df["category"].nunique()}')
print(f'[OK] Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print("\nSample:")
print(df.head())

## Cell 4 - Shared helpers

In [ ]:
# This cell defines the main file paths, model save paths, and shared settings.
# We keep them in one place so later cells stay consistent and easier to edit.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report

print("Loading enhanced helpers.")

def save_json(data, path, show_progress=True):
    """Save JSON with progress feedback"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if show_progress:
        print(f"Saving to {path}.")

    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

    size_mb = path.stat().st_size / 1024**2
    if show_progress:
        print(f"[OK] Saved {path} ({size_mb:.1f} MB)")

def load_splits(progress=True):
    """Load splits with progress bars"""
    files = [TRAIN_PATH, VAL_PATH, TEST_PATH]
    names = ['train', 'val', 'test']

    if progress:
        print("Verifying files.")
        for name, path in zip(names, files):
            status = '[OK]' if path.exists() else '[ERROR]'
            print(f"  {status} {name}: {path}")

    assert all(p.exists() for p in files), f"Missing files: {[p for p in files if not p.exists()]}"

    if progress:
        print("\nLoading with progress.")

    datasets = {}
    for name, path in tqdm(zip(names, files), total=3, desc="Loading splits"):
        datasets[name] = pd.read_csv(path, dtype={'text':'string'})

    if progress:
        total_rows = sum(len(df) for df in datasets.values())
        print(f"[OK] Loaded {total_rows:,} total rows")

    return datasets['train'], datasets['val'], datasets['test']

def precision_at_k_from_proba(y_true, y_proba, classes, k_values=(1, 3, 5)):
    """Precision@K with progress"""
    print(f"Computing P@{k_values}.")
    y_true = pd.Series(y_true).reset_index(drop=True)
    classes = np.array(classes)
    results = {}

    for k in tqdm(k_values, desc="P@K"):
        top_k_indices = np.argsort(y_proba, axis=1)[:, -k:][:, ::-1]
        correct = sum(1 for i, true_label in enumerate(y_true)
                     if np.where(classes == true_label)[0][0] in top_k_indices[i])
        results[f'P@{k}'] = correct / len(y_true)

    return results

def precision_at_k_from_logits(logits, y_true, k_values=(1, 3, 5)):
    """Torch logits version with progress"""
    print("Computing GPU P@K.")
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    y_true = np.array(y_true)
    results = {}

    for k in tqdm(k_values, desc="Logits P@K"):
        top_k = np.argsort(probs, axis=1)[:, -k:]
        correct = sum(1 for i, true_idx in enumerate(y_true) if true_idx in top_k[i])
        results[f'P@{k}'] = correct / len(y_true)

    return results

print('[OK] Enhanced helpers loaded!')
print("Features: Progress bars, file checks, memory stats")

## Cell 5 - Optional validation of merged dataset
This is safe to rerun and does not change files.

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import numpy as np

print("Dataset Explorer with Progress.")

# Loading with progress bar
print(f"Loading {MERGED_PATH.name}.")
chunksize = 50000
chunks = []
chunk_bars = tqdm(pd.read_csv(MERGED_PATH, chunksize=chunksize), desc="Chunks", unit="50k rows")

for chunk in chunk_bars:
    chunks.append(chunk)
    chunk_bars.set_postfix(rows=len(chunk))

df = pd.concat(chunks, ignore_index=True)
print(f'[OK] Loaded {len(df):,} rows across {df["category"].nunique()} categories')
print(f'[OK] Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

print('\n' + '=' * 70)
print('TOP 3 SUBREDDITS PER CATEGORY')
print('=' * 70)

# Progress for category analysis
top_cats = df['category'].value_counts().head(10).index
print(f"Analyzing top {len(top_cats)} categories.")

for i, cat in enumerate(tqdm(top_cats, desc="Categories")):
    cat_df = df[df['category'] == cat]
    top_subs = cat_df['original_subreddit'].value_counts().head(3)
    print(f'\n{cat}:')
    for sub, count in top_subs.items():
        print(f'  {sub:<25} ({count:,} posts)')

print('\n' + '=' * 70)
print('VALIDATION STATS')
print('=' * 70)

cat_counts = df['category'].value_counts()
print(f'Total posts:           {len(df):>8,}')
print(f'Categories:           {df["category"].nunique():>8}')
print(f'Avg posts/category:    {len(df) // df["category"].nunique():>8,}')
print(f'Most balanced:        {cat_counts.min():>8,}')
print(f'Most posts:            {cat_counts.max():>8,} ({cat_counts.index[0]})')
print(f'Text length:           {df["text"].str.len().mean():.0f} ± {df["text"].str.len().std():.0f} chars')
print(f'Null values:           {df.isnull().sum().sum():>8,}')

print('\n[OK] Dataset validated!')

## Cell 6 - Create stratified train/validation/test split
This starts from `reddit_merged_categories.csv` only.

In [ ]:
# This cell checks that the merged dataset exists and loads it in chunks.
# We do this to confirm the file is ready and to avoid memory issues while reading a large CSV.

from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
from tqdm import tqdm

FORCE_RESPLIT = False

print("Dataset Splitter with Progress.")

# File check
files_exist = all(p.exists() for p in [TRAIN_PATH, VAL_PATH, TEST_PATH])
if files_exist and not FORCE_RESPLIT:
    print('Using existing splits:')
    for p in tqdm([TRAIN_PATH, VAL_PATH, TEST_PATH], desc="Loading"):
        name = p.name.replace('.csv', '').title()
        globals()[name.lower()] = pd.read_csv(p)
    print("[OK] Existing splits loaded!")
else:
    assert MERGED_PATH.exists(), f'Missing {MERGED_PATH}'
    print(f"Loading full dataset {MERGED_PATH.name}.")

    # Progress loading
    chunks = []
    for chunk in tqdm(pd.read_csv(MERGED_PATH, chunksize=50000), desc="Loading", unit="chunk"):
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    print(f"[OK] Full dataset: {df.shape}")

    print(f"Categories: {df['category'].nunique()}")
    print('\nCategory distribution:')
    cat_dist = df['category'].value_counts()
    print(cat_dist)

    # Progress splitting
    print("\nSplitting dataset.")
    train_temp, test = train_test_split(
        df, test_size=0.15, stratify=df['category'], random_state=RANDOM_STATE
    )
    print(f"[OK] Train+Val: {len(train_temp):,} | Test: {len(test):,}")

    print("Step 2/2: Train (74%) vs Val (11%)")
    train, val = train_test_split(
        train_temp, test_size=0.1765, stratify=train_temp['category'], random_state=RANDOM_STATE
    )
    print(f"[OK] Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

    # Save with progress
    print("\nSaving splits.")
    for data, path, name in [(train, TRAIN_PATH, "Train"), (val, VAL_PATH, "Val"), (test, TEST_PATH, "Test")]:
        data.to_csv(path, index=False)
        print(f"[OK] {name}.csv saved ({len(data):,} rows)")

print('\nFinal Shapes:')
print(f'Val:       {val.shape}')
print(f'Test:      {test.shape}')

# Distribution check with progress
print('\nNormalized Distributions:')
for data, name in [(train, "Train"), (val, "Val"), (test, "Test")]:
    print(f'\n{name}:')
    dist = data['category'].value_counts(normalize=True).round(4)
    print(dist.head(5))
    print(dist)

print('\n[OK] Split complete & validated!')
print('Files:')
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(f'  {p}')

## Cell 7 - Naive Bayes setup

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

train, val, test = load_splits()
print(f'Train: {train.shape[0]:,} samples')
print(f'Val: {val.shape[0]:,} samples')
print(f'Test: {test.shape[0]:,} samples')
print(f'Categories: {train["category"].nunique()}')

## Cell 8 - Train Naive Bayes

In [ ]:
# This cell creates helper functions used across training and evaluation steps.
# We do this to avoid repeating code for saving files, loading splits, and computing metrics.

import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np
from pathlib import Path
from scipy.sparse import vstack

import nltk
from nltk.corpus import stopwords

# Define ALL missing constants
NB_MODEL_PATH = Path("/content/drive/MyDrive/InformationRet/nb_model.pkl")
print(f"[OK] Saved to Drive: {NB_MODEL_PATH}")

FORCE_RETRAIN_NB = True  # Set False after first run

print("Training Naive Bayes.")

train, val, test = load_splits()

if NB_MODEL_PATH.exists() and not FORCE_RETRAIN_NB:
    print("Loading model.")
    pipeline_nb = joblib.load(NB_MODEL_PATH)
    print(f"[OK] Loaded {NB_MODEL_PATH}")
else:
    print("Starting NB training.")

    try:
        stopwords.words('english')
    except LookupError:
        nltk.download('stopwords')
    
    english_stop_words = set(stopwords.words('english'))
    
    custom_stop_words = english_stop_words.union({
        "don't", "it's", "i'm", "you're", "he's", "she's", "we're", "they're",
        "i've", "you've", "we've", "they've", "i'll", "you'll", "he'll", "she'll",
        "we'll", "they'll", "i'd", "you'd", "he'd", "she'd", "we'd", "they'd",
        "can't", "won't", "wouldn't", "shouldn't", "couldn't", "mustn't", "haven't",
        "hasn't", "hadn't", "isn't", "aren't", "wasn't", "weren't", "doesn't",
        "didn't", "lb" # Explicitly add 'lb' to stop words
    })

    texts = train['text'].fillna('').str.replace('\n', ' ', regex=False).str.replace('lb lb', ' ', regex=False).str.replace('lb', ' ', regex=False).values

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10000,
        min_df=5,
        max_df=0.8,
        lowercase=True,
        stop_words=list(custom_stop_words), # Use the custom stop words list
        dtype=np.float32,
        token_pattern=r"\b\w+(?:'\w+)*\b" # Corrected to handle apostrophes
    )
    
    print("Fitting vocabulary.")
    tfidf.fit(texts)
    print(f"[OK] Vocab: {len(tfidf.vocabulary_):,} words")

    print("Transforming text to features.")
    batch_size = 50000
    X_train = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Transform"):
        batch_X = tfidf.transform(texts[i:i+batch_size])
        X_train.append(batch_X)

    # Combine sparse matrices
    X_train = vstack(X_train, format='csr')
    print(f"[OK] Features: {X_train.shape}")

    nb = MultinomialNB(alpha=1.0)
    print("Fitting Naive Bayes model.")
    nb.fit(X_train, train['category'].values)

    pipeline_nb = Pipeline([('tfidf', tfidf), ('nb', nb)])

    print("Saving model.")
    joblib.dump(pipeline_nb, NB_MODEL_PATH, compress=3)
    size = NB_MODEL_PATH.stat().st_size / 1024**2
    print(f"[OK] Saved {size:.1f}MB: {NB_MODEL_PATH}")

# Benchmark
print("\nTesting model.")
val_pred = pipeline_nb.predict(val['text'][:5000])
print(f"Val acc (5k): {accuracy_score(val['category'][:5000], val_pred):.4f}")

print("Model training complete.")

## Cell 9 - Evaluate Naive Bayes

In [ ]:
val_pred = pipeline_nb.predict(val['text'])
val_proba = pipeline_nb.predict_proba(val['text'])
test_pred = pipeline_nb.predict(test['text'])
test_proba = pipeline_nb.predict_proba(test['text'])
TOP_K_VALUES = [1, 3, 5]  # From Cell 2

print('\nValidation Results:')
print(classification_report(val['category'], val_pred, zero_division=0))

val_metrics_nb = precision_at_k_from_proba(val['category'], val_proba, pipeline_nb.classes_, TOP_K_VALUES)
test_metrics_nb = precision_at_k_from_proba(test['category'], test_proba, pipeline_nb.classes_, TOP_K_VALUES)

print('\n[OK] Precision@K (Validation):')
for metric, score in val_metrics_nb.items():
    print(f'{metric}: {score:.4f}')

print('\n[OK] Precision@K (Test):')
for metric, score in test_metrics_nb.items():
    print(f'{metric}: {score:.4f}')

# Replace your last block with this:
METRICS_DIR = Path('/content/drive/MyDrive/InformationRet/metrics')
METRICS_DIR.mkdir(exist_ok=True)
save_json({
    'validation_metrics': val_metrics_nb,
    'test_metrics': test_metrics_nb
}, METRICS_DIR / 'nb_metrics.json')
print("[OK] nb_metrics.json saved!")

## Cell 11 - Train Logistic Regression

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.sparse import vstack
from tqdm import tqdm
import os

import nltk
from nltk.corpus import stopwords

LR_MODEL_PATH = Path("/content/drive/MyDrive/InformationRet/lr_model.pkl")
TRAIN_PATH = Path("/content/drive/MyDrive/InformationRet/train.csv")
VAL_PATH = Path("/content/drive/MyDrive/InformationRet/val.csv")
TEST_PATH = Path("/content/drive/MyDrive/InformationRet/test.csv")

RANDOM_STATE = 42
FORCE_RETRAIN_LR = True

print("Training Logistic Regression.")

for name, path in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} file: {path}")
    print(f"[OK] {name}: {path}")

train = pd.read_csv(
    TRAIN_PATH,
    low_memory=False,
    dtype={"text": "string"},
    memory_map=True
)
val = pd.read_csv(
    VAL_PATH,
    low_memory=False,
    dtype={"text": "string"},
    memory_map=True
)
test = pd.read_csv(
    TEST_PATH,
    low_memory=False,
    dtype={"text": "string"},
    memory_map=True
)

for name, df in [("train", train), ("val", val), ("test", test)]:
    if "text" not in df.columns or "category" not in df.columns:
        raise ValueError(f"{name} is missing required columns: text/category")

print(f"Train: {train.shape[0]:,} | Val: {val.shape[0]:,} | Test: {test.shape[0]:,}")
print(f"Categories: {train['category'].nunique()}")

if LR_MODEL_PATH.exists() and not FORCE_RETRAIN_LR:
    pipeline_lr = joblib.load(LR_MODEL_PATH)
    print(f"[OK] Loaded existing LR model from {LR_MODEL_PATH}")
else:
    print("Starting LR training.")

    try:
        stopwords.words('english')
    except LookupError:
        nltk.download('stopwords')

    # Add custom stop words, including common contractions and 'lb'
    custom_stop_words = english_stop_words.union({
        "don't", "it's", "i'm", "you're", "he's", "she's", "we're", "they're",
        "i've", "you've", "we've", "they've", "i'll", "you'll", "he'll", "she'll",
        "we'll", "they'll", "i'd", "you'd", "he'd", "she'd", "we'd", "they'd",
        "can't", "won't", "wouldn't", "shouldn't", "couldn't", "mustn't", "haven't",
        "hasn't", "hadn't", "isn't", "aren't", "wasn't", "weren't", "doesn't",
        "didn't", "lb" # Explicitly add 'lb' to stop words
    })

    texts = train["text"].fillna('').str.replace('\n', ' ', regex=False).str.replace('lb lb', ' ', regex=False).str.replace('lb', ' ', regex=False).values
    y_train = train["category"].values

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=8000,
        min_df=5,
        max_df=0.8,
        lowercase=True,
        stop_words=list(custom_stop_words), # Use the custom stop words list
        dtype=np.float32,
        token_pattern=r"\b\w+(?:'\w+)*\b" # Corrected to handle apostrophes
    )

    print("Fitting vocabulary.")
    tfidf.fit(texts)
    print(f"[OK] Vocab: {len(tfidf.vocabulary_):,}")

    print("Transforming text to features.")
    batch_size = 50000
    X_train_parts = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
        batch_texts = texts[i:i + batch_size]
        X_batch = tfidf.transform(batch_texts)
        X_train_parts.append(X_batch)

    X_train = vstack(X_train_parts, format="csr")
    print(f"[OK] X_train: {X_train.shape}")

    lr = LogisticRegression(
        solver="saga",
        multi_class="auto",
        max_iter=500,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )

    print("Fitting Logistic Regression model.")
    lr.fit(X_train, y_train)

    pipeline_lr = Pipeline([
        ("tfidf", tfidf),
        ("lr", lr)
    ])

    joblib.dump(pipeline_lr, LR_MODEL_PATH, compress=3)
    print(f"[OK] Saved {LR_MODEL_PATH.stat().st_size / 1024**2:.1f}MB")

print("\nLogistic Regression model trained.")
val_preview = val["text"].fillna("").astype(str).iloc[:5000]
val_pred = pipeline_lr.predict(val_preview)
print(f"{accuracy_score(val['category'].iloc[:5000], val_pred):.1%}")

## Cell 12 - Evaluate Logistic Regression

In [ ]:
# This cell creates the train, validation, and test splits from the merged dataset.
# We do this so each model is trained and evaluated on separate data.

from IPython.display import clear_output
clear_output(wait=True)
val_proba_lr = pipeline_lr.predict_proba(val['text'])
val_pred_lr = pipeline_lr.predict(val['text'])
test_proba_lr = pipeline_lr.predict_proba(test['text'])
test_pred_lr = pipeline_lr.predict(test['text'])

# Now call the function - it should work!
p_at_k_val_lr = precision_at_k_from_proba(val['category'], val_proba_lr, pipeline_lr.classes_)
p_at_k_test_lr = precision_at_k_from_proba(test['category'], test_proba_lr, pipeline_lr.classes_)

print('\nValidation Results:')
print(classification_report(val['category'], val_pred_lr, zero_division=0))
print('\n[OK] Precision@K (Validation):')
for metric, score in p_at_k_val_lr.items():
    print(f'{metric}: {score:.4f}')

print('\n[OK] Precision@K (Test):')
for metric, score in p_at_k_test_lr.items():
    print(f'{metric}: {score:.4f}')

import json

# Define METRICS_DIR (adjust path as needed)
BASE_DIR = Path('/content/drive/MyDrive/InformationRet')
METRICS_DIR = BASE_DIR / 'metrics'
METRICS_DIR.mkdir(exist_ok=True)

save_json({
    'validation_metrics': p_at_k_val_lr,
    'test_metrics': p_at_k_test_lr
}, METRICS_DIR / 'lr_metrics.json')

## Cell 13 - Transformer environment check

In [ ]:
import transformers
import torch
import os

print('VERSION:', transformers.__version__)
print('PATH:', transformers.__file__)
print('CPU cores:', os.cpu_count())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1024**3)
    torch.backends.cudnn.benchmark = True
else:
    print('No GPU detected (training will be slow)')

torch.set_num_threads(os.cpu_count())

## Cell 14 - DistilBERT label prep

In [ ]:
# This cell loads the saved dataset splits and imports the tools needed for Naive Bayes.
# We do this to prepare the baseline model setup before training.

from transformers import DistilBertTokenizerFast
from datasets import Dataset as HFDataset
import pandas as pd
import numpy as np

# PATHS
TRAIN_PATH = '/content/drive/MyDrive/InformationRet/train.csv'
VAL_PATH = '/content/drive/MyDrive/InformationRet/val.csv'
TEST_PATH = '/content/drive/MyDrive/InformationRet/test.csv'

print("Loading data.")

# Load data with python engine
train_df = pd.read_csv(TRAIN_PATH, dtype={'text':'string'}, engine='python')
val_df = pd.read_csv(VAL_PATH, dtype={'text':'string'}, engine='python')
test_df = pd.read_csv(TEST_PATH, dtype={'text':'string'}, engine='python')

# Label encoding
all_labels = sorted(train_df['category'].unique())
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

train_df['category'] = train_df['category'].map(label2id)
val_df['category'] = val_df['category'].map(label2id)
test_df['category'] = test_df['category'].map(label2id)

num_labels = len(all_labels)
print(f'[OK] Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')
print(f'[OK] Labels: {num_labels}')
print("DistilBERT ready.")

## Cell 15 - DistilBERT tokenize and cache

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import DistilBertTokenizerFast
from pathlib import Path
import pandas as pd
from tqdm import tqdm

print("DistilBERT Tokenization.")

BASE_DIR = Path("/content/drive/MyDrive/InformationRet")
DISTIL_TOKENIZED_DIR = BASE_DIR / "distilbert_tokenized"
DISTIL_TOKENIZED_DIR.mkdir(exist_ok=True, parents=True)

# Load
train_df = pd.read_csv(BASE_DIR / "train.csv", dtype={'text': 'string'})
val_df = pd.read_csv(BASE_DIR / "val.csv", dtype={'text': 'string'})
test_df = pd.read_csv(BASE_DIR / "test.csv", dtype={'text': 'string'})

print(f"Loaded: train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}")

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors=None  # Avoid tensor crash
    )

print("Tokenizing text.")

# Train (batched=32k chunks)
print("Train.")
train_ds = Dataset.from_pandas(train_df[['text', 'category']])
with tqdm(total=len(train_df), desc="Train") as pbar:
    def update_pbar(*args, **kwargs):
        pbar.update(32000)  # Batch size estimate
        return tokenize_function(*args, **kwargs)
    train_ds = train_ds.map(
        update_pbar,
        batched=True,
        batch_size=32000,  # Stable batch size
        desc="Train"
    )

# Val
print("Val.")
val_ds = Dataset.from_pandas(val_df[['text', 'category']])
val_ds = val_ds.map(
    tokenize_function,
    batched=True,
    batch_size=32000,
    desc="Val"
)

# Test
print("Test.")
test_ds = Dataset.from_pandas(test_df[['text', 'category']])
# PASTE THIS ONLY (your variables exist)
print("Formatting and saving datasets.")
for ds_name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"Formatting {ds_name}.")
    ds.rename_column('category', 'labels')
    if all(c in ds.column_names for c in ['input_ids', 'attention_mask', 'labels']):
        ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
        print(f"[OK] {ds_name} formatted")

tokenized_distil = DatasetDict({
    'train': train_ds, 'validation': val_ds, 'test': test_ds
})

tokenized_distil.save_to_disk(str(DISTIL_TOKENIZED_DIR))
print(f"[OK] SAVED: {DISTIL_TOKENIZED_DIR}")

print("DistilBERT preparation complete.")

## Cell 16 - DistilBERT train

In [ ]:
# This cell trains the Naive Bayes text classifier and saves the model.
# We do this to build a simple baseline that is fast and easy to compare against later models.

import torch
import numpy as np
from transformers import (
    DistilBertForSequenceClassification, DistilBertTokenizerFast,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import load_from_disk, ClassLabel, Value
from pathlib import Path
import joblib

BASE_DIR = Path('/content/drive/MyDrive/InformationRet')
DISTIL_TOKENIZED_DIR = BASE_DIR / "distilbert_tokenized"
DISTIL_RESULTS_DIR = BASE_DIR / "distilbert_results"
DISTIL_MODEL_DIR = DISTIL_RESULTS_DIR / "best_model"
DISTIL_RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Load datasets
tokenized_distil = load_from_disk(str(DISTIL_TOKENIZED_DIR))
train_ds_distil = tokenized_distil['train'].rename_column('category', 'labels')
val_ds_distil = tokenized_distil['validation'].rename_column('category', 'labels')
test_ds_distil = tokenized_distil['test'].rename_column('category', 'labels')

# Convert labels to integers
label_list = sorted(set(train_ds_distil['labels']))
id2label = {i: str(label) for i, label in enumerate(label_list)}
label2id = {str(label): i for i, label in enumerate(label_list)}

def convert_labels(example):
    example['labels'] = label2id[example['labels']]
    return example

train_ds_distil = train_ds_distil.map(convert_labels)
val_ds_distil = val_ds_distil.map(convert_labels)
test_ds_distil = test_ds_distil.map(convert_labels)

# Clean columns
train_ds_distil = train_ds_distil.remove_columns(['text'])
val_ds_distil = val_ds_distil.remove_columns(['text'])
test_ds_distil = test_ds_distil.remove_columns(['text'])

print(f"[OK] Labels fixed: {train_ds_distil.column_names}")
print(f"[OK] Sample labels: {train_ds_distil['labels'][:3]}")

# Model + Tokenizer
tokenizer_distil = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model_distil = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# TrainingArgs
training_args_distil = TrainingArguments(
    output_dir=str(DISTIL_RESULTS_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=300,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='steps',
    save_steps=2000,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=2000,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    dataloader_num_workers=0,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    dataloader_pin_memory=False
)

# Simple DataCollator (no padding issues)
class SimpleDataCollator:
    def __call__(self, features):
        batch = {}
        for key in features[0].keys():
            batch[key] = torch.stack([torch.tensor(f[key]) for f in features])
        return batch

data_collator = SimpleDataCollator()
trainer_distil = Trainer(
    model=model_distil,
    args=training_args_distil,
    train_dataset=train_ds_distil,
    eval_dataset=val_ds_distil,
    data_collator=data_collator
)

# Train
FORCE_RETRAIN_DISTIL = False
if DISTIL_MODEL_DIR.exists() and not FORCE_RETRAIN_DISTIL:
    print(f'[OK] Using existing: {DISTIL_MODEL_DIR}')
else:
    print("Training DistilBERT.")
    trainer_distil.train()
    trainer_distil.save_model(str(DISTIL_MODEL_DIR))
    tokenizer_distil.save_pretrained(str(DISTIL_MODEL_DIR))
    print(f"[OK] SAVED: {DISTIL_MODEL_DIR}")

print("DistilBERT training complete.")
joblib.dump({'id2label': id2label, 'label2id': label2id}, DISTIL_RESULTS_DIR / 'labels.pkl')

## Cell 17 - DistilBERT evaluate

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import torch

from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification, DistilBertTokenizerFast,
    TrainingArguments, Trainer
)

# Define paths
BASE_DIR = Path('/content/drive/MyDrive/InformationRet')
DISTIL_RESULTS_DIR = BASE_DIR / "distilbert_results"
# Use the "best_model" directory which is saved by the trainer
DISTIL_MODEL_DIR = DISTIL_RESULTS_DIR / "best_model"

# Check if model directory exists
if not DISTIL_MODEL_DIR.exists():
    raise FileNotFoundError(f"Trained DistilBERT model not found at {DISTIL_MODEL_DIR}. Please ensure training was completed successfully (Cell 16).")

print("Loading trained DistilBERT model and tokenizer...")
tokenizer_distil = DistilBertTokenizerFast.from_pretrained(DISTIL_MODEL_DIR)
model_distil = DistilBertForSequenceClassification.from_pretrained(DISTIL_MODEL_DIR)

# Load label mapping
label_mapping = joblib.load(DISTIL_RESULTS_DIR / "labels.pkl")
label2id = label_mapping['label2id']
id2label = label_mapping['id2label']

# Load dataframes
val_df = pd.read_csv(BASE_DIR / "val.csv", dtype={'text':'string'}, engine='python')
test_df = pd.read_csv(BASE_DIR / "test.csv", dtype={'text':'string'}, engine='python')

# Helper function to prepare dataset for prediction
def prepare_hf_dataset(df, tokenizer, label2id):
    ds = Dataset.from_pandas(df[['text', 'category']])
    ds = ds.map(
        lambda ex: tokenizer(ex['text'], truncation=True, padding='max_length', max_length=256),
        batched=True, batch_size=1000,
        desc="Tokenizing & Mapping Labels"
    )
    ds = ds.rename_column('category', 'labels').remove_columns(['text'])
    ds = ds.map(lambda ex: {'labels': label2id[str(ex['labels'])]})
    # Set format to torch explicitly for trainer prediction
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds

print("Preparing validation and test datasets...")
val_ds = prepare_hf_dataset(val_df, tokenizer_distil, label2id)
test_ds = prepare_hf_dataset(test_df, tokenizer_distil, label2id)

# Initialize Trainer for prediction (minimal arguments)
# output_dir is required, even if not saving anything
training_args = TrainingArguments(
    output_dir=str(DISTIL_RESULTS_DIR), # Required, but actual output won't be used for prediction
    per_device_eval_batch_size=16, # Use a reasonable batch size for evaluation
    report_to='none', # Prevent logging to external services
    fp16=torch.cuda.is_available(), # Enable if GPU is available
    dataloader_num_workers=0 # Set to 0 for single process during evaluation for stability
)

trainer_distil = Trainer(
    model=model_distil,
    args=training_args,
)

# Make predictions
print("Generating validation predictions")
val_predictions_distil = trainer_distil.predict(val_ds)
val_accuracy_distil = np.mean(val_predictions_distil.predictions.argmax(-1) == val_predictions_distil.label_ids)
print(f'Validation accuracy = {val_accuracy_distil:.1%}')

print("Generating test predictions")
predictions_distil = trainer_distil.predict(test_ds)  # This variable name matches original f9a787f6
accuracy_distil = np.mean(predictions_distil.predictions.argmax(-1) == predictions_distil.label_ids)
print(f'Test accuracy = {accuracy_distil:.1%}')

In [ ]:
# This cell evaluates the Naive Bayes model on validation and test data.
# We do this to measure accuracy and top-k performance and save the results.

import numpy as np
from sklearn.metrics import top_k_accuracy_score

# Proper Precision@K using sklearn
TOP_K_VALUES = [1, 3, 5]
print('DISTILBERT TEST RESULTS')
print(f'Accuracy@1  = {accuracy_distil:.1%}')

for k in TOP_K_VALUES:
    p_at_k = top_k_accuracy_score(predictions_distil.label_ids,
                                predictions_distil.predictions,
                                k=k)
    print(f'Precision@{k} = {p_at_k:.1%}')

In [ ]:
# See which model trainer_distil uses
print("Trainer model name:", trainer_distil.model.__class__.__name__)
print("Best checkpoint:", trainer_distil.state.best_model_checkpoint)
print("Current eval results:", trainer_distil.state.log_history[-1] if trainer_distil.state.log_history else "No eval history")

# List saved checkpoints
import os
checkpoints = [d for d in os.listdir('.') if d.startswith('checkpoint-')]
print("Available checkpoints:", checkpoints)

In [ ]:
# Print ALL categories first
print("ALL CATEGORIES:")
print(sorted(id2label.values()))

# Check label distribution in test set (confirms correct mapping)
print("\nTest label distribution:")
unique, counts = np.unique(predictions_distil.label_ids, return_counts=True)
for label, count in zip(unique, counts):
    print(f"{id2label[label]}: {count}")